# Post-processing add-on к прогону `dinos224_geo2_v5`

**Что это.** Надстройка над уже обученным прогоном. Ни одна модель здесь не обучается заново:
используются сохранённые OOF- и тест-вероятности (`*_fold*_preds.joblib`), а два необязательных
блока (§4 и §5) только **делают инференс** уже сохранёнными весами.

**Как запускать.** Двумя способами:

1. *В той же сессии* — выполнить основной ноутбук до §11 включительно, затем эти ячейки.
   Все переменные (`P_oof`, `P_te`, `Y_use`, `fit_rule`, …) подхватятся из памяти.
2. *С нуля* — выполнить основной ноутбук до §6 (фолды) включительно, затем эти ячейки:
   источники будут прочитаны с диска.

---

## Что показал реальный прогон

| величина | значение |
|---|---|
| CNN, 5 фолдов, OOF | **14.250** |
| GBM на геометрии, OOF | 12.961 |
| весь пайплайн (5 источников + правило), OOF | **14.474** |
| вложенная (честная) оценка | **14.124 ± 0.322** |
| лидерборд | **14.987** |
| бутстрап той же модели на 669 объектах | 14.230 ± 0.385, 95% ИД [13.45, **14.95**] |

**Главный вывод из этой таблицы.** 14.987 лежит на самой верхней границе 95%-го интервала,
который ноутбук сам же и посчитал для своей OOF-модели. То есть 14.99 — это не «модель лучше,
чем показал OOF», а удачная выборка теста. Честная оценка текущего решения — около **14.1–14.5**,
и планировать следующий шаг надо от неё, а не от 14.99.

**Второй вывод, менее приятный.** Вложенная оценка (14.124) **ниже**, чем CNN с простыми
порогами (14.250). Весь второй уровень — GBM, стэкинг, powerset, по-классовый бленд — на
настроенных-и-проверенных-на-одних-данных числах даёт +0.22, а на честной оценке не даёт
ничего. Разрыв 0.35 балла — это чистое переобучение решающего правила. Это и есть самый
дешёвый резерв: он не требует ни одной эпохи обучения. Им занимается §2.


In [ ]:
# ============================ §0. Подключение к прогону ============================
import os, gc, json, math, time, itertools, warnings
from pathlib import Path
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, matplotlib as mpl
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')

mpl.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#fbfbfd', 'axes.grid': True,
                     'grid.alpha': .25, 'grid.linestyle': '--', 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10, 'axes.titlesize': 11,
                     'axes.titleweight': 'bold'})
PAL = {'good': '#2e9e5b', 'bad': '#d1495b', 'warn': '#e8a33d', 'main': '#2d6cdf',
       'alt': '#8a4fbd', 'grey': '#9aa0a6', 'dark': '#1f2933'}


class Rep:
    W = 82
    @classmethod
    def section(cls, t, s=''):
        print('\n' + '=' * cls.W); print(f'  {t}')
        if s: print(f'  {s}')
        print('=' * cls.W)
    @staticmethod
    def kv(**kw):
        for k, v in kw.items(): print(f'    {k:<34s} {v}')
    @staticmethod
    def ok(m): print(f'  [+] {m}')
    @staticmethod
    def warn(m): print(f'  [!] {m}')
    @staticmethod
    def bar(v, lo=10, hi=20, w=36, label=''):
        f = 0 if hi <= lo else max(0., min(1., (v - lo) / (hi - lo)))
        print(f'    {label:<22s} |{"█" * int(round(f * w))}{"·" * (w - int(round(f * w)))}| {v:.3f}')


R = Rep
R.section('Подключение к прогону')

SESSION = all(n in globals() for n in ('P_oof', 'P_te', 'Y_use', 'SOURCES', 'folds_use'))
if SESSION:
    R.ok('переменные найдены в текущей сессии — читать с диска не нужно')
else:
    R.warn('переменных в сессии нет — собираю источники с диска')
    assert 'CFG' in globals() and 'train_df' in globals() and 'fold' in train_df.columns, \
        'Сначала выполните основной ноутбук до §6 (фолды) включительно.'
    ART = CFG.work / 'artifacts'
    FEAT_VER_ = globals().get('FEAT_VER', 'v4selfint')

    def _load(name):
        p = ART / f'{name}.joblib'
        return joblib.load(p) if p.exists() else None

    parts = [_load(f'{CFG.run_tag}_fold{k}_preds') for k in range(CFG.n_folds)]
    assert all(p is not None for p in parts), 'не найдены *_fold*_preds.joblib — проверьте CFG.run_tag'
    o = pd.DataFrame(np.nan, index=train_df['item_id'], columns=CFG.target_cols)
    te_acc = 0
    for d in parts:
        o.loc[d['oof_ids'], CFG.target_cols] = d['oof']
        te_acc = te_acc + pd.DataFrame(d['te'], columns=CFG.target_cols,
                                       index=d['te_ids']) / CFG.n_folds
    cnn_oof = o.values
    cnn_te = te_acc.reindex(test_df['item_id']).values
    use = ~np.isnan(cnn_oof[:, 0])

    SOURCES, P_oof, P_te = ['CNN'], [cnn_oof[use]], [cnn_te]
    for nm, key in [('GBM', f'gbm_geometry_{FEAT_VER_}'), ('STACK', f'{CFG.run_tag}_stack_{FEAT_VER_}'),
                    ('POWERSET', f'{CFG.run_tag}_powerset_{FEAT_VER_}'), ('DINO', f'{CFG.run_tag}_dino_gbm')]:
        d = _load(key)
        if d is None:
            R.warn(f'{nm}: артефакт {key} не найден, источник пропущен'); continue
        oo = np.asarray(d['oof']); tt = np.asarray(d['te'])
        P_oof.append(oo[use] if len(oo) == len(use) else oo)
        P_te.append(tt); SOURCES.append(nm)
    Y_use = train_df.loc[use, CFG.target_cols].values
    folds_use = train_df.loc[use, 'fold'].values

Y_use = np.asarray(Y_use); folds_use = np.asarray(folds_use)
W_CLS = Y_use[:, :10].sum(0) / Y_use[:, :10].sum()
R.kv(**{'источники': ', '.join(SOURCES), 'объектов OOF': len(Y_use),
        'объектов теста': len(P_te[0])})


def metric(y, p):
    return (10 * f1_score(y[:, 10], p[:, 10], zero_division=0)
            + 10 * f1_score(y[:, :10], p[:, :10], average='weighted', zero_division=0))


## §1. Что именно принесло 14.99 — разбор по вкладам

Считаем каждый источник в одиночку и нарастающим итогом, плюс раскладываем метрику
по классам: сколько баллов получено и сколько ещё лежит на столе.


In [ ]:
R.section('§1. Разбор вкладов')

def exact_thr(y, p, plateau=0.995):
    # Точный argmax F1 за один проход. Разрез допустим только там, где значение p
    # МЕНЯЕТСЯ: иначе порог попадает между двумя одинаковыми вероятностями, и `p > th`
    # выбрасывает оба объекта вместо одного. При насыщенных сигмоидах и усреднении
    # по фолдам совпадения массовые, так что это не теоретическая тонкость.
    y = np.asarray(y).astype(np.int32); p = np.asarray(p, float)
    P = int(y.sum())
    if P == 0: return 0.5, 0.0
    o = np.argsort(-p, kind='stable'); ys, ps = y[o], p[o]
    f1 = 2.0 * np.cumsum(ys) / (np.arange(1, len(y) + 1) + P)
    fb = float(f1.max())
    cut = np.empty(len(ps), bool); cut[-1] = True; cut[:-1] = ps[:-1] > ps[1:]
    ok = np.where((f1 >= plateau * fb) & cut)[0]
    if not len(ok): ok = np.where(cut)[0][[int(np.argmax(f1[cut]))]]
    i = int(ok[len(ok) // 2])
    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < len(ps) else ps[i] - 1e-6), fb


def simple_score(P):
    th = np.array([exact_thr(Y_use[:, i], P[:, i])[0] for i in range(11)])
    return metric(Y_use, (P > th).astype(int))


solo = {nm: simple_score(P) for nm, P in zip(SOURCES, P_oof)}
pcf = {nm: pd.Series([f1_score(Y_use[:, i],
                               (P[:, i] > exact_thr(Y_use[:, i], P[:, i])[0]).astype(int),
                               zero_division=0) for i in range(11)], index=CFG.target_cols)
       for nm, P in zip(SOURCES, P_oof)}
F1M = pd.DataFrame(pcf)

base = F1M['CNN']
pts = pd.DataFrame({'F1': base.round(3),
                    'вес': np.append(W_CLS, np.nan).round(4),
                    'получено': np.append(10 * W_CLS * base.values[:10], 10 * base['quality']).round(3),
                    'потолок': np.append(10 * W_CLS, 10.0).round(3)}, index=CFG.target_cols)
pts['резерв'] = (pts['потолок'] - pts['получено']).round(3)

fig = plt.figure(figsize=(16, 9)); gs = fig.add_gridspec(2, 3, hspace=.38, wspace=.3)

ax = fig.add_subplot(gs[0, 0])
s = pd.Series(solo).sort_values()
ax.barh(s.index, s.values, color=[PAL['grey']] * (len(s) - 1) + [PAL['main']])
for i, v in enumerate(s.values): ax.text(v + .05, i, f'{v:.2f}', va='center', fontsize=9)
ax.set_xlim(11, 15.4); ax.set_title('Каждый источник в одиночку (OOF)')

ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(F1M.values, cmap='RdYlGn', vmin=.1, vmax=.9, aspect='auto')
ax.set_xticks(range(F1M.shape[1])); ax.set_xticklabels(F1M.columns, rotation=30, ha='right', fontsize=8)
ax.set_yticks(range(11)); ax.set_yticklabels(CFG.target_cols, fontsize=8); ax.grid(False)
for i in range(11):
    for j in range(F1M.shape[1]):
        ax.text(j, i, f'{F1M.values[i, j]:.2f}', ha='center', va='center', fontsize=6.5)
ax.set_title('F1 по классам: специализация источников')

ax = fig.add_subplot(gs[0, 2])
b = pts.drop(index='quality').sort_values('потолок')
y = np.arange(len(b))
ax.barh(y, b['потолок'], color='#e3e6ea', label='потолок')
ax.barh(y, b['получено'], color=PAL['main'], label='получено')
ax.set_yticks(y); ax.set_yticklabels(b.index, fontsize=8)
for i, (g, c) in enumerate(zip(b['получено'], b['потолок'])):
    ax.text(c + .03, i, f'+{c - g:.2f}', va='center', fontsize=7.5, color=PAL['grey'])
ax.set_xlim(0, b['потолок'].max() * 1.4); ax.legend(fontsize=8, loc='lower right')
ax.set_title('Дефекты: получено против потолка')

ax = fig.add_subplot(gs[1, 0])
res = pts['резерв'].sort_values(ascending=False)
cols = [PAL['bad'] if n == 'quality' else PAL['warn'] for n in res.index]
ax.bar(range(len(res)), res.values, color=cols)
ax.set_xticks(range(len(res))); ax.set_xticklabels(res.index, rotation=75, fontsize=8)
ax.set_ylabel('баллов'); ax.set_title(f'Где лежат недостающие баллы (всего {res.sum():.2f})')

ax = fig.add_subplot(gs[1, 1])
lvl = {'CNN\n(OOF)': 14.250, 'весь\nпайплайн': 14.474, 'вложенная\n(честная)': 14.124,
       'лидер-\nборд': 14.987, 'цель': 16.0}
cols = [PAL['main'], PAL['main'], PAL['warn'], PAL['good'], PAL['bad']]
ax.bar(range(5), list(lvl.values()), color=cols)
ax.errorbar([3], [14.987], yerr=[[.385], [.385]], fmt='none', ecolor=PAL['dark'], capsize=6)
for i, v in enumerate(lvl.values()): ax.text(i, v + .07, f'{v:.2f}', ha='center', fontsize=9, weight='bold')
ax.set_xticks(range(5)); ax.set_xticklabels(lvl, fontsize=8)
ax.set_ylim(13.5, 16.5); ax.set_title('Уровни, о которых идёт речь')

ax = fig.add_subplot(gs[1, 2]); ax.axis('off'); ax.grid(False)
txt = ['Три факта из прогона:', '',
       '1. 14.99 на ЛБ — верхняя граница 95% ИД',
       '   [13.45, 14.95] собственного бутстрапа.',
       '   Реальный уровень решения ≈ 14.1–14.5.', '',
       '2. Вложенная оценка 14.12 НИЖЕ, чем CNN',
       '   с простыми порогами (14.25): весь второй',
       '   уровень честного прироста не даёт.',
       '   Разрыв 0.35 — переобучение правила (§2).', '',
       f'3. Половина резерва — quality ({pts.loc["quality", "резерв"]:.2f} б.),',
       '   он ломается от любой ложной сработки.']
for i, t in enumerate(txt):
    ax.text(0, 1 - .072 * i, t, fontsize=9, va='top', weight='bold' if t.endswith(':') else 'normal')
plt.show()

display(pts.sort_values('резерв', ascending=False))
print('\nисточники в одиночку:', {k: round(v, 3) for k, v in solo.items()})


## §2. Главный бесплатный резерв: правило перестало переобучаться

Разрыв «настроено и проверено на одних данных» (14.474) против вложенной оценки (14.124) —
0.35 балла. Он возникает из трёх мест, и все три лечатся без обучения:

1. **По-классовые веса бленда для редких классов.** У `intersection` и `scale` по 107 позитивов;
   выбор лучшей из 81 комбинации весов по такой выборке — это подгонка под единичные объекты.
2. **Одна-единственная настройка порогов на всём OOF.** Порог — оценка по выборке, у неё есть
   дисперсия. Усреднение порогов по пяти подвыборкам (bagging) снижает её, ничего не стоя.
3. **Сдвиг распределений OOF против теста.** Вероятность OOF — предсказание **одной** модели
   (той, в чьём валидационном фолде объект оказался), а тестовая — среднее **пяти**. Среднее
   менее разбросано, поэтому один и тот же абсолютный порог отсекает на тесте другую долю
   объектов. Лечится переходом от абсолютного порога к целевой доле положительных (§3).

Ниже честное сравнение: каждый вариант правила оценивается **вложенно** — настраивается на
четырёх фолдах, проверяется на пятом. Только эти числа сопоставимы с лидербордом.


In [ ]:
R.section('§2. Правило: бэггинг + ограничение по-классовых весов')

def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps); return np.log(p / (1 - p))


def to_rank(p):
    p = np.asarray(p, float)
    if p.ndim == 1: p = p[:, None]; sq = True
    else: sq = False
    out = np.empty_like(p); n = len(p)
    for j in range(p.shape[1]):
        o = np.argsort(p[:, j], kind='stable'); r = np.empty(n); r[o] = np.arange(n)
        out[:, j] = r / max(n - 1, 1)
    return out[:, 0] if sq else out


def combine(parts, w, space):
    w = np.asarray(w, float)
    w = np.ones(len(parts)) if w.sum() <= 0 else w / w.sum()
    if space == 'prob': return sum(wi * p for wi, p in zip(w, parts))
    if space == 'rank': return sum(wi * to_rank(p) for wi, p in zip(w, parts))
    return 1 / (1 + np.exp(-sum(wi * to_logit(p) for wi, p in zip(w, parts))))


_WG = {}
def wgrid(k, coarse=False):
    key = (k, coarse)
    if key not in _WG:
        step = (0, .5, 1.) if (coarse or k > 3) else (0, .25, .5, .75, 1.)
        _WG[key] = [w for w in itertools.product(step, repeat=k) if sum(w) > 0]
    return _WG[key]


def fit_rule2(parts, Y, space=None, min_support=300, coarse=False):
    spaces = ['prob', 'logit', 'rank'] if space is None else [space]
    grid = wgrid(len(parts), coarse)
    best = None
    for sp in spaces:
        sup = Y[:, :10].sum(0).astype(float); wc = sup / max(sup.sum(), 1)
        gb, gw = -1, tuple([1] * len(parts))
        for w in grid:
            pb = combine(parts, w, sp)
            v = sum(wc[i] * exact_thr(Y[:, i], pb[:, i])[1] for i in range(10))
            if v > gb: gb, gw = v, w
        W = np.tile(np.asarray(gw, float), (11, 1))
        for i in range(11):
            if int(Y[:, i].sum()) < min_support: continue
            cb, cw = -1, gw
            for w in grid:
                v = exact_thr(Y[:, i], combine([p[:, [i]] for p in parts], w, sp)[:, 0])[1]
                if v > cb: cb, cw = v, w
            W[i] = cw
        prob = np.column_stack([combine([p[:, [i]] for p in parts], W[i], sp)[:, 0] for i in range(11)])
        th = np.array([exact_thr(Y[:, i], prob[:, i])[0] for i in range(10)])
        pd_ = (prob[:, :10] > th).astype(int)
        fa = f1_score(Y[:, :10], pd_, average='weighted', zero_division=0)
        soft = np.prod(1 - np.clip(prob[:, :10], 0, 1), 1)
        hard = (pd_.sum(1) == 0).astype(float)
        qb = (-1, .5, .5, 'mix')
        for a in np.arange(0, 1.01, .05):
            pq = a * prob[:, 10] + (1 - a) * soft
            t, v = exact_thr(Y[:, 10], pq)
            if v > qb[0]: qb = (v, float(a), float(t), 'mix')
        for mode in ('and', 'or'):
            for a in np.arange(0, 1.01, .1):
                pq = a * prob[:, 10] + (1 - a) * soft
                t, _ = exact_thr(Y[:, 10], pq)
                q = ((pq > t) * hard if mode == 'and' else np.maximum((pq > t), hard))
                v = f1_score(Y[:, 10], q.astype(int), zero_division=0)
                if v > qb[0]: qb = (v, float(a), float(t), mode)
        tot = 10 * qb[0] + 10 * fa
        if best is None or tot > best['oof']:
            best = {'space': sp, 'W': W, 'th': th, 'q_alpha': qb[1], 'q_th': qb[2],
                    'q_mode': qb[3], 'oof': tot, 'rate': pd_.mean(0)}
    return best


def apply_rule2(parts, Rl, rates=None):
    # rates: если передать целевые доли положительных, пороги пересчитываются как квантили
    # распределения ИМЕННО этой выборки (см. §3).
    W = np.asarray(Rl['W'])
    prob = np.column_stack([combine([p[:, [i]] for p in parts], W[i], Rl['space'])[:, 0]
                            for i in range(11)])
    if rates is None:
        th = Rl['th']
    else:
        th = np.array([np.quantile(prob[:, i], 1 - min(max(rates[i], 1e-4), .999))
                       for i in range(10)])
    pd_ = (prob[:, :10] > th).astype(int)
    soft = np.prod(1 - np.clip(prob[:, :10], 0, 1), 1)
    hard = (pd_.sum(1) == 0).astype(int)
    pq = Rl['q_alpha'] * prob[:, 10] + (1 - Rl['q_alpha']) * soft
    qh = (pq > Rl['q_th']).astype(int)
    q = {'mix': qh, 'and': qh * hard, 'or': np.maximum(qh, hard)}[Rl['q_mode']]
    return np.concatenate([pd_, q[:, None]], 1), prob


def bag_rules(Rs):
    # Усреднение нескольких правил: веса и пороги — среднее, стратегия quality — большинство.
    out = dict(Rs[0])
    out['W'] = np.mean([np.asarray(r['W']) for r in Rs], 0)
    out['th'] = np.mean([r['th'] for r in Rs], 0)
    out['q_alpha'] = float(np.mean([r['q_alpha'] for r in Rs]))
    out['q_th'] = float(np.mean([r['q_th'] for r in Rs]))
    modes = [r['q_mode'] for r in Rs]
    out['q_mode'] = max(set(modes), key=modes.count)
    spaces = [r['space'] for r in Rs]
    out['space'] = max(set(spaces), key=spaces.count)
    out['rate'] = np.mean([r['rate'] for r in Rs], 0)
    return out


def fit_bagged(parts, Y, folds, space=None, min_support=300, coarse=False):
    Rs = []
    for k in np.unique(folds):
        m = folds != k
        Rs.append(fit_rule2([p[m] for p in parts], Y[m], space, min_support, coarse))
    sp = max(set(r['space'] for r in Rs), key=[r['space'] for r in Rs].count)
    Rs = [r if r['space'] == sp else
          fit_rule2([p[folds != k] for p in parts], Y[folds != k], sp, min_support, coarse)
          for r, k in zip(Rs, np.unique(folds))]
    return bag_rules(Rs)


def nested(fit_fn, parts, Y, folds):
    # Честная оценка: правило настраивается на 4 фолдах, метрика считается на 5-м.
    sc = []
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        Rk = fit_fn([p[tr] for p in parts], Y[tr], folds[tr])
        pk, _ = apply_rule2([p[va] for p in parts], Rk)
        sc.append(metric(Y[va], pk))
    return np.array(sc)


VAR = {
    'как в v5 (по-классовые веса, min_sup=300)':
        lambda P, Y, f: fit_rule2(P, Y, min_support=300),
    'только глобальные веса':
        lambda P, Y, f: fit_rule2(P, Y, min_support=10 ** 9),
    'грубая сетка весов':
        lambda P, Y, f: fit_rule2(P, Y, min_support=300, coarse=True),
    'бэггинг правила по 4 подвыборкам':
        lambda P, Y, f: fit_bagged(P, Y, f, min_support=300),
    'бэггинг + только глобальные веса':
        lambda P, Y, f: fit_bagged(P, Y, f, min_support=10 ** 9),
    'бэггинг + глобальные + грубая сетка':
        lambda P, Y, f: fit_bagged(P, Y, f, min_support=10 ** 9, coarse=True),
}

res = {}
for nm, fn in VAR.items():
    t0 = time.time()
    s = nested(fn, P_oof, Y_use, folds_use)
    res[nm] = (s.mean(), s.std(), time.time() - t0)
    print(f'  {nm:<44s} вложенная {s.mean():6.3f} ± {s.std():.3f}   ({time.time() - t0:.0f} c)')

RES = pd.DataFrame(res, index=['вложенная', 'разброс', 'сек']).T.sort_values('вложенная',
                                                                            ascending=False)
BEST_NAME = RES.index[0]
RULE2 = VAR[BEST_NAME](P_oof, Y_use, folds_use)
pred2, prob2 = apply_rule2(P_oof, RULE2)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
r = RES.sort_values('вложенная')
cols = [PAL['good'] if i == len(r) - 1 else PAL['grey'] for i in range(len(r))]
ax[0].barh(range(len(r)), r['вложенная'], xerr=r['разброс'], color=cols, capsize=4)
ax[0].set_yticks(range(len(r))); ax[0].set_yticklabels(r.index, fontsize=8)
ax[0].axvline(14.124, color=PAL['bad'], ls='--', lw=1.2)
ax[0].text(14.13, -.4, 'v5 = 14.12', color=PAL['bad'], fontsize=8)
ax[0].set_xlim(13.6, max(r['вложенная']) + .35)
ax[0].set_title('Вложенная (честная) метрика вариантов правила')

ax[1].bar(['v5:\nна OOF', 'v5:\nвложенно', f'новое:\nвложенно'],
          [14.474, 14.124, RES.iloc[0]['вложенная']],
          color=[PAL['grey'], PAL['warn'], PAL['good']])
for i, v in enumerate([14.474, 14.124, RES.iloc[0]['вложенная']]):
    ax[1].text(i, v + .03, f'{v:.3f}', ha='center', fontsize=10, weight='bold')
ax[1].set_ylim(13.8, 15.0); ax[1].set_title('Сколько отыграно на переобучении правила')
plt.tight_layout(); plt.show()

R.kv(**{'лучший вариант': BEST_NAME,
        'вложенная метрика': f"{RES.iloc[0]['вложенная']:.3f} (было 14.124)",
        'прирост': f"{RES.iloc[0]['вложенная'] - 14.124:+.3f} балла, ноль обучения",
        'пространство бленда': RULE2['space'],
        'quality': f"{RULE2['q_mode']}, alpha={RULE2['q_alpha']:.2f}"})


## §3. Сдвиг распределений OOF против теста

Вероятность на OOF даёт **одна** модель, на тесте — среднее **пяти**. Среднее всегда менее
разбросано, поэтому порог, настроенный на OOF, на тесте отсекает другую долю объектов.
Эффект односторонний и предсказуемый: доля предсказанных дефектов на тесте оказывается
ниже, а `quality` — завышенным.

Проверяем это прямо: сравниваем стандартное отклонение вероятностей и долю положительных.
Если расхождение подтверждается, переходим от абсолютного порога к **целевой доле**
положительных: порог на тесте берётся как квантиль его собственного распределения.

**Насколько это важно.** На симуляции, воспроизводящей условия прогона (8962 объекта OOF по
одной модели, 669 тестовых по среднему пяти, реальные доли классов), переход к порогам по доле
дал **+1.06 балла**, тогда как бэггинг правила в тех же условиях не дал ничего. Разброс
вероятностей при усреднении пяти моделей сжимался в 0.55 раза — ровно тот механизм, что описан
выше.

Оговорка, которую важно держать в голове: в симуляции пять моделей независимы, а реальные
модели пяти фолдов обучены на перекрывающихся данных и потому коррелированы — сжатие будет
слабее, и прирост тоже. Поэтому ячейка ниже **измеряет фактическое сжатие** на ваших числах
и включает механизм только если он подтверждается.


In [ ]:
R.section('§3. Сдвиг распределений и пороги по доле')

pred_te_abs, prob_te = apply_rule2(P_te, RULE2)
rate_oof = pred2.mean(0)
rate_te_abs = pred_te_abs.mean(0)
prior = Y_use.mean(0)

std_oof = prob2.std(0); std_te = prob_te.std(0)
shrink = float(np.median(std_te / (std_oof + 1e-9)))

cmp = pd.DataFrame({'доля в train': prior.round(3), 'доля OOF': rate_oof.round(3),
                    'доля test (абс. порог)': rate_te_abs.round(3),
                    'std OOF': std_oof.round(3), 'std test': std_te.round(3)},
                   index=CFG.target_cols)
cmp['сжатие std'] = (cmp['std test'] / cmp['std OOF']).round(3)
display(cmp)

# пороги по целевой доле; для редких классов доля на 669 объектах сама шумная,
# поэтому смешиваем абсолютный порог и квантильный пополам
# Целевая доля = доля, которую правило даёт на OOF. Для редких классов оценка доли по
# 669 объектам сама шумная (у intersection это ~8 объектов), поэтому подмешиваем априорную
# долю из train тем сильнее, чем меньше ожидаемое число положительных на тесте.
n_exp = rate_oof[:10] * len(P_te[0])
lam = np.clip(n_exp / (n_exp + 25.0), 0, 1)          # мало ожидаемых -> опираемся на train
rates_target = lam * rate_oof[:10] + (1 - lam) * prior[:10]
pred_te_q, _ = apply_rule2(P_te, RULE2, rates=rates_target)
pred_te_q[:, 10] = pred_te_abs[:, 10]                # quality решается своей стратегией

# консервативный вариант: метка ставится только если согласны оба порога
pred_te_mix = ((pred_te_abs.astype(float) + pred_te_q.astype(float)) >= 1.5).astype(int)
pred_te_mix[:, 10] = pred_te_abs[:, 10]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
x = np.arange(11)
ax[0].bar(x - .27, prior, .27, label='train', color=PAL['dark'])
ax[0].bar(x, rate_oof, .27, label='OOF', color=PAL['main'])
ax[0].bar(x + .27, rate_te_abs, .27, label='test (абс.)', color=PAL['warn'])
ax[0].set_xticks(x); ax[0].set_xticklabels(CFG.target_cols, rotation=80, fontsize=8)
ax[0].legend(fontsize=8); ax[0].set_title('Доля предсказанных единиц')

ax[1].bar(range(11), (cmp['std test'] / cmp['std OOF']).values, color=PAL['alt'])
ax[1].axhline(1, color=PAL['dark'], ls='--')
ax[1].set_xticks(range(11)); ax[1].set_xticklabels(CFG.target_cols, rotation=80, fontsize=8)
ax[1].set_title(f'Сжатие разброса вероятностей (медиана {shrink:.2f})')

d = rate_te_abs[:10] - rate_oof[:10]
ax[2].barh(CFG.artifact_cols, d, color=[PAL['bad'] if v < 0 else PAL['good'] for v in d])
ax[2].axvline(0, color=PAL['dark'])
ax[2].set_title('test минус OOF по доле дефектов')
plt.tight_layout(); plt.show()

if shrink < 0.90:
    PRED_TE = pred_te_q
    R.warn(f'разброс на тесте сжат в {1 / shrink:.2f} раза — абсолютные пороги смещены заметно; '
           f'перехожу на пороги по целевой доле')
elif shrink < 0.97:
    PRED_TE = pred_te_mix
    R.warn(f'сжатие умеренное ({shrink:.2f}) — беру консервативный вариант: '
           f'метка ставится только при согласии обоих порогов')
else:
    PRED_TE = pred_te_abs
    R.ok(f'сжатия практически нет ({shrink:.2f}) — оставляю абсолютные пороги')

ch = int((PRED_TE != pred_te_abs).sum())
print(f'\nизменено меток против абсолютного порога: {ch} из {PRED_TE.size} '
      f'({100 * ch / PRED_TE.size:.2f}%)')
print('доли дефектов после коррекции:')
print(pd.DataFrame({'OOF': rate_oof[:10].round(3), 'test до': rate_te_abs[:10].round(3),
                    'test после': PRED_TE.mean(0)[:10].round(3)},
                   index=CFG.artifact_cols).T.to_string())


## §4. (опционально, ~20 мин) Расширенный TTA поверх сохранённых весов

Обучения нет: берём уже сохранённые `{run_tag}_cnn_fold{k}.pth` и делаем инференс с
бóльшим числом преобразований.

Ключевое преобразование — **циклический сдвиг четырёх азимутальных ракурсов**. Повороту
объекта на 90° соответствует простое переименование видов 0→1→2→3→0; множество из шести
картинок не меняется, значит **все 11 меток инвариантны**. Это не «ротация, убивающая
`partial`», от которой отказался baseline, а точная симметрия съёмки.

Оговорка: модель из v5 (общий бэкбон + attention-pooling) сама по себе инвариантна к порядку
видов, поэтому от циклического сдвига **в чистом виде эффекта не будет** — он даёт прирост
только в сочетании с флипом и лёгким масштабированием, которые порядок не затрагивают.
Ячейка меряет вклад каждого преобразования отдельно, так что решение включать его или нет
принимается по числу, а не на веру.


In [ ]:
RUN_EXTRA_TTA = False     # <- поставьте True, если готовы потратить ~20 минут

if RUN_EXTRA_TTA:
    import torch
    from torch.utils.data import DataLoader
    R.section('§4. Расширенный TTA на сохранённых весах')
    need = ['MultiViewNet', 'MeshViewDataset', 'feat_tr', 'feat_te', 'FEAT_COLS',
            'IMG_TRAIN', 'IMG_TEST', 'train_df', 'test_df', '_to_fp32']
    miss = [n for n in need if n not in globals()]
    assert not miss, f'нет переменных {miss}: выполните основной ноутбук до §9 включительно'

    AZ, PO = [0, 1, 2, 3], [4, 5]
    def perm_of(s): return [AZ[(k + s) % 4] for k in range(4)] + PO

    @torch.no_grad()
    def predict_tta(model, loader, transforms):
        model.eval(); out, ids = [], []
        from tqdm.auto import tqdm as _t
        for b in _t(loader, desc=f'TTA x{len(transforms)}', leave=False):
            v = b['views'].to(CFG.device); f = b['feats'].to(CFG.device)
            acc = 0
            with torch.cuda.amp.autocast(enabled=CFG.amp):
                for s, flip, sc in transforms:
                    vv = v[:, perm_of(s)]
                    if flip: vv = torch.flip(vv, dims=[-1])
                    if sc != 1.0:
                        B, N, C, H, Wd = vv.shape
                        vv = torch.nn.functional.interpolate(
                            vv.flatten(0, 1), scale_factor=sc, mode='bilinear',
                            align_corners=False)
                        vv = torch.nn.functional.interpolate(
                            vv, size=(H, Wd), mode='bilinear', align_corners=False).view(B, N, C, H, Wd)
                    acc = acc + torch.sigmoid(model(vv, f)).float()
            out.append((acc / len(transforms)).cpu().numpy()); ids.extend(b['item_id'])
        return np.vstack(out), ids

    SETS = {'база (как в v5): вид + флип': [(0, False, 1.), (0, True, 1.)],
            '+ циклические сдвиги (x8)': [(s, fl, 1.) for s in range(4) for fl in (False, True)],
            '+ масштаб 0.9/1.0 (x4)': [(0, fl, sc) for fl in (False, True) for sc in (.9, 1.)],
            'всё вместе (x16)': [(s, fl, sc) for s in range(4) for fl in (False, True)
                                 for sc in (.9, 1.)]}

    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)
    scores, oof_new, te_new = {}, {}, {}
    for nm, tf in SETS.items():
        O = np.full((len(train_df), 11), np.nan); T = 0
        for k in range(CFG.n_folds):
            wp = CFG.work / f'{CFG.run_tag}_cnn_fold{k}.pth'
            if not wp.exists():
                R.warn(f'нет весов {wp.name} — пропускаю набор'); O = None; break
            model.load_state_dict(_to_fp32(torch.load(wp, map_location='cpu', weights_only=False)))
            va = train_df[train_df.fold == k].reset_index(drop=True)
            lva = DataLoader(MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False),
                             batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers)
            lte = DataLoader(MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}),
                                             IMG_TEST, feat_te, train=False, labels=False),
                             batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers)
            op, oid = predict_tta(model, lva, tf)
            tp, _ = predict_tta(model, lte, tf)
            idx = train_df.set_index('item_id').index.get_indexer(oid)
            O[idx] = op; T = T + tp / CFG.n_folds
        if O is None: continue
        m = ~np.isnan(O[:, 0])
        scores[nm] = simple_score(O[m])
        oof_new[nm], te_new[nm] = O[m], T
        print(f'  {nm:<32s} OOF {scores[nm]:.3f}')

    if scores:
        best_tta = max(scores, key=scores.get)
        fig, ax = plt.subplots(figsize=(9, 3.6))
        s = pd.Series(scores)
        ax.bar(range(len(s)), s.values,
               color=[PAL['good'] if i == s.values.argmax() else PAL['grey'] for i in range(len(s))])
        ax.set_xticks(range(len(s))); ax.set_xticklabels(s.index, rotation=20, ha='right', fontsize=8)
        ax.set_ylim(min(s) - .1, max(s) + .1); ax.set_title('Вклад наборов TTA (OOF)')
        for i, v in enumerate(s.values): ax.text(i, v + .01, f'{v:.3f}', ha='center', fontsize=9)
        plt.tight_layout(); plt.show()
        if scores[best_tta] > scores.get('база (как в v5): вид + флип', -1) + 0.01:
            P_oof[0] = oof_new[best_tta]; P_te[0] = te_new[best_tta]
            R.ok(f'источник CNN заменён на «{best_tta}» (+{scores[best_tta] - solo["CNN"]:.3f})')
            RULE2 = VAR[BEST_NAME](P_oof, Y_use, folds_use)
            pred2, prob2 = apply_rule2(P_oof, RULE2)
            PRED_TE, _ = apply_rule2(P_te, RULE2, rates=pred2.mean(0)[:10])
        else:
            R.warn('расширенный TTA прироста не дал — оставляю базовый')
else:
    R.warn('§4 выключен (RUN_EXTRA_TTA = False)')


## §5. (опционально, ~25 мин) Замороженный DINOv3 как ещё один источник

Обучения основной сети снова нет — только прогон 6 видов через замороженный бэкбон и
лёгкий LightGBM поверх эмбеддингов.

Почему именно это. Правило отбора в ансамбль — **не сила источника, а некоррелированность
его ошибок**. Замороженный бэкбон меток не видел вообще, поэтому не повторяет систематических
промахов дообученной сети. В прогоне v5 эту роль играл DINOv2 ViT-B/14; DINOv3 обучен на
1.689B изображений против 142M и в timm ≥ 1.0.20 доступен как `vit_base_patch16_dinov3`.


In [ ]:
RUN_DINOV3 = False        # <- True, если готовы потратить ~25 минут (нужен timm>=1.0.20)

if RUN_DINOV3:
    R.section('§5. Замороженный DINOv3 -> LightGBM')
    import torch, timm, lightgbm as lgb
    from sklearn.decomposition import PCA
    from tqdm.auto import tqdm as _t
    tv = tuple(int(x) for x in timm.__version__.split('.')[:3])
    assert tv >= (1, 0, 20), f'timm {timm.__version__}: нужен >= 1.0.20. !pip install -U "timm>=1.0.20"'

    MODEL_NAME = 'vit_base_patch16_dinov3.lvd1689m'
    TL = 224

    @torch.no_grad()
    def emb(ids, img_dir, split):
        m = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0, img_size=TL)
        m = m.to(CFG.device).eval()
        mean = torch.tensor(MEAN, device=CFG.device).view(1, 3, 1, 1)
        std = torch.tensor(STD, device=CFG.device).view(1, 3, 1, 1)
        out, bs = [], 16
        for s0 in _t(range(0, len(ids), bs), desc=f'DINOv3 {split}'):
            ch = ids[s0:s0 + bs]; batch = []
            for iid in ch:
                p = Path(img_dir) / f'{iid}.png'
                v = load_views(p, CFG.tile) if p.exists() else np.full((6, CFG.tile, CFG.tile, 3), 255, np.uint8)
                batch.append(v)
            x = torch.from_numpy(np.stack(batch)).to(CFG.device)
            B = x.shape[0]
            x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, CFG.tile, CFG.tile).float() / 255.
            if CFG.tile != TL:
                x = torch.nn.functional.interpolate(x, size=(TL, TL), mode='bilinear', align_corners=False)
            x = (x - mean) / std
            with torch.cuda.amp.autocast(enabled=CFG.amp):
                z = m(x).float().view(B, 6, -1)
            out.append(torch.cat([z.mean(1), z.max(1).values], 1).cpu().numpy())
        del m; gc.collect(); torch.cuda.empty_cache()
        return np.vstack(out).astype(np.float32)

    Etr = cached(f'{CFG.run_tag}_dinov3_emb_train',
                 lambda: emb(train_df['item_id'].tolist(), IMG_TRAIN, 'train'), compress=1)
    Ete = cached(f'{CFG.run_tag}_dinov3_emb_test',
                 lambda: emb(test_df['item_id'].tolist(), IMG_TEST, 'test'), compress=1)
    pca = PCA(n_components=min(160, Etr.shape[1]), random_state=CFG.seed).fit(np.vstack([Etr, Ete]))
    Ptr, Pte_ = pca.transform(Etr), pca.transform(Ete)
    R.kv(**{'эмбеддинг': Etr.shape[1], 'PCA': f'{Ptr.shape[1]} компонент, '
            f'{pca.explained_variance_ratio_.sum():.1%} дисперсии'})

    Yall = train_df[CFG.target_cols].values; fall = train_df['fold'].values
    oof_d3 = np.zeros((len(Ptr), 11)); te_d3 = np.zeros((len(Pte_), 11))
    for i, col in enumerate(CFG.target_cols):
        for k in range(CFG.n_folds):
            tri, vai = np.where(fall != k)[0], np.where(fall == k)[0]
            spw = max((Yall[tri, i] == 0).sum() / max((Yall[tri, i] == 1).sum(), 1), 1.)
            m = lgb.LGBMClassifier(objective='binary', learning_rate=.03, num_leaves=31,
                                   min_child_samples=30, feature_fraction=.7, bagging_fraction=.8,
                                   bagging_freq=1, n_estimators=700, verbosity=-1, n_jobs=-1,
                                   seed=CFG.seed, scale_pos_weight=min(spw, 20.))
            m.fit(Ptr[tri], Yall[tri, i], eval_set=[(Ptr[vai], Yall[vai, i])],
                  eval_metric='average_precision',
                  callbacks=[lgb.early_stopping(60, verbose=False)])
            oof_d3[vai, i] = m.predict_proba(Ptr[vai])[:, 1]
            te_d3[:, i] += m.predict_proba(Pte_)[:, 1] / CFG.n_folds

    o3 = oof_d3[use] if 'use' in globals() and len(oof_d3) == len(use) else oof_d3
    print(f'  DINOv3 frozen OOF: {simple_score(o3):.3f}')
    e_new = np.abs(o3 - Y_use); e_cnn = np.abs(P_oof[0] - Y_use)
    rr = np.mean([np.corrcoef(e_cnn[:, i], e_new[:, i])[0, 1] for i in range(11)])
    print(f'  корреляция ошибок с дообученной сетью: {rr:.2f} '
          f'({"хорошая диверсификация" if rr < .5 else "источники похожи"})')

    P_oof.append(o3); P_te.append(te_d3); SOURCES.append('DINOv3')
    s_new = nested(VAR[BEST_NAME], P_oof, Y_use, folds_use)
    print(f'  вложенная метрика с новым источником: {s_new.mean():.3f} ± {s_new.std():.3f}')
    if s_new.mean() > RES.iloc[0]['вложенная']:
        RULE2 = VAR[BEST_NAME](P_oof, Y_use, folds_use)
        pred2, prob2 = apply_rule2(P_oof, RULE2)
        PRED_TE, _ = apply_rule2(P_te, RULE2, rates=pred2.mean(0)[:10])
        R.ok(f'источник принят (+{s_new.mean() - RES.iloc[0]["вложенная"]:.3f})')
    else:
        P_oof.pop(); P_te.pop(); SOURCES.pop()
        R.warn('источник прироста не дал — откатываю')
else:
    R.warn('§5 выключен (RUN_DINOV3 = False)')


## §6. Новый сабмит и итог

In [ ]:
R.section('§6. Итог')

sub = pd.DataFrame(PRED_TE, columns=CFG.target_cols)
sub.insert(0, 'item_id', test_df['item_id'].values)
sub = sub[['item_id'] + CFG.target_cols]
assert len(sub) == len(test_df) and sub['item_id'].is_unique
sub.to_csv('submission_postproc.csv', index=False)

final_nested = float(RES.iloc[0]['вложенная'])
R.kv(**{'источники': ', '.join(SOURCES),
        'правило': f"{BEST_NAME} | {RULE2['space']} | quality={RULE2['q_mode']}",
        'вложенная оценка (была 14.124)': f'{final_nested:.3f}',
        'прирост без единой эпохи обучения': f'{final_nested - 14.124:+.3f}',
        'файл': 'submission_postproc.csv'})
R.bar(14.124, 13.5, 16.5, label='v5, вложенная')
R.bar(final_nested, 13.5, 16.5, label='после post-proc')
R.bar(16.0, 13.5, 16.5, label='цель')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
lv = {'v5\nвложенная': 14.124, 'v5\nлидерборд': 14.987, 'post-proc\nвложенная': final_nested,
      'цель': 16.0}
ax[0].bar(range(4), list(lv.values()),
          color=[PAL['warn'], PAL['grey'], PAL['good'], PAL['bad']])
for i, v in enumerate(lv.values()): ax[0].text(i, v + .04, f'{v:.2f}', ha='center', weight='bold')
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(lv, fontsize=8)
ax[0].set_ylim(13.5, 16.4); ax[0].set_title('Где мы сейчас')

x = np.arange(11)
ax[1].bar(x - .2, Y_use.mean(0), .4, label='train', color=PAL['dark'])
ax[1].bar(x + .2, np.append(PRED_TE.mean(0)[:10], PRED_TE[:, 10].mean()), .4,
          label='новый сабмит', color=PAL['good'])
ax[1].set_xticks(x); ax[1].set_xticklabels(CFG.target_cols, rotation=80, fontsize=8)
ax[1].legend(fontsize=8); ax[1].set_title('Доли меток в сабмите против train')
plt.tight_layout(); plt.show()

tail = [
    '',
    'Что дальше, если нужно 16.',
    '  Post-processing выжимает переобучение решающего правила и сдвиг распределений —',
    '  это реальные, но ограниченные 0.2-0.5 балла. Оставшийся разрыв до 16 закрывается',
    '  только вторым ИСТОЧНИКОМ СИГНАЛА, а не новой обработкой тех же вероятностей:',
    '',
    '  1. Второй прогон другим бэкбоном (convnext_small.dinov3 @ 256 или',
    '     vit_base_patch16_dinov3), новый run_tag, затем CFG.ensemble_tags = [оба].',
    '     Ожидаемо +0.3...0.6. Обучать заново нужно ТОЛЬКО новую конфигурацию —',
    '     текущая остаётся как есть и участвует готовыми предсказаниями.',
    '  2. Число эпох: лог прогона показывает, что лучшая эпоха приходится на 4-9 из 16,',
    '     дальше метрика падает. 10-12 эпох дадут тот же результат вдвое быстрее —',
    '     это освобождает время ровно под пункт 1.',
]
print(chr(10).join(tail))
display(sub.head())
